# СимургФарм — углублённый EDA и проверка биологических формул

**Данные:** `expamlomycin_train.csv`, `expamlomycin_test.csv`. Это отдельный исследовательский ноутбук для Google Colab, а не новая версия итогового submission.

Цели: (1) проверить, какие физиологические признаки математически избыточны; (2) сравнить альтернативные формулы функции почек; (3) оценить признаки распределения жидкости, воспаления и взаимодействия; (4) принять решения по одинаковой 5-fold CV **без использования тестовых ответов**.

⚠️ Экзампломицин — препарат конкурсной постановки. Ни одна из исследованных формул не предназначена для расчёта дозы реальным пациентам.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, QuantileRegressor
import statsmodels.api as sm

if not (Path('expamlomycin_train.csv').exists() and Path('expamlomycin_test.csv').exists()):
    from google.colab import files
    print('Загрузите оба файла: train и test')
    files.upload()
train = pd.read_csv('expamlomycin_train.csv')
test = pd.read_csv('expamlomycin_test.csv', sep=';')
print('Train:', train.shape, 'Test:', test.shape)
display(train.head(3))

## 1. Формулы из литературы и гипотезы

- **Cockcroft–Gault** — оценка клиренса креатинина из возраста, массы тела, пола и SCr.
- **CKD-EPI 2021** — современная оценка СКФ по креатинину, возрасту и полу, без коэффициента расы. Для сравнения используем как индексированную величину, так и абсолютную (`× BSA/1.73`).
- **Devine IBW и AdjBW** — альтернативы фактической массе для оценки функции почек; не являются универсальной заменой веса.
- **Watson TBW** — оценка общей воды организма на основе возраста/роста/массы/пола. Это **не** объём распределения лекарственного препарата, а лишь гипотетически связанный дескриптор.
- **Janmahasatian LBW** — формула безжировой массы из пола, веса, роста через BMI.
- **Взаимодействия** — масса × сепсис, альбумин × сепсис и др.

Первоисточники: [NIDDK CKD-EPI 2021](https://www.niddk.nih.gov/research-funding/research-programs/kidney-clinical-research-epidemiology/laboratory/glomerular-filtration-rate-equations/adults), [NIDDK и BSA](https://www.niddk.nih.gov/research-funding/research-programs/kidney-clinical-research-epidemiology/laboratory/ckd-drug-dosing-providers), [Janmahasatian](https://pmc.ncbi.nlm.nih.gov/articles/PMC3833312/), [Watson (1980)](https://ajcn.nutrition.org/article/S0002-9165%2823%2943745-8/pdf), [обзор PopPK аминогликозидов](https://pmc.ncbi.nlm.nih.gov/articles/PMC8145041/).

In [ ]:
def add_bio_features(data):
    d = data.copy()
    w = d['weight_kg']
    h = d['height_cm']
    age = d['age']
    scr = d['scr_umol_l']
    female = (d['sex'] == 'F')
    male = ~female
    sep = d['sepsis_status']
    d['female'] = female.astype(int)
    d['oat_rapid'] = (d['oat_variant'] == 'Rapid').astype(int)
    d['oat_slow'] = (d['oat_variant'] == 'Slow').astype(int)
    d['BMI'] = w / (h / 100)**2
    d['BSA'] = np.sqrt(h*w/3600)  # Mosteller
    d['Janma_LBW'] = 9270*w / np.where(male, 6680 + 216*d['BMI'], 8780 + 244*d['BMI'])
    d['Watson_TBW'] = np.where(male,
        2.447 - 0.09516*age + 0.1074*h + 0.3362*w,
        -2.097 + 0.1069*h + 0.2466*w)
    d['IBW'] = np.maximum(25, np.where(male, 50.0, 45.5) + 0.9*(h - 152.4))
    d['AdjBW'] = np.where(w > d['IBW'], d['IBW'] + .4*(w - d['IBW']), w)
    def cg(used_weight):
        return ((140 - age)*used_weight / (0.814*scr))*np.where(female, .85, 1.)
    d['CG_actual'] = cg(w)
    d['CG_lean'] = cg(d['lean_bw_kg'])
    d['CG_adjusted'] = cg(d['AdjBW'])
    scr_mg_dl = scr / 88.4
    kappa = np.where(female, .7, .9)
    alpha = np.where(female, -.241, -.302)
    z = scr_mg_dl/kappa
    d['CKDEPI_indexed'] = 142*np.minimum(z, 1)**alpha*np.maximum(z, 1)**(-1.2)*(.9938**age)*np.where(female, 1.012, 1.)
    d['CKDEPI_absolute'] = d['CKDEPI_indexed']*d['BSA']/1.73
    for name in ['CG_actual','CG_lean','CG_adjusted','CKDEPI_indexed','CKDEPI_absolute']:
        d['log_'+name] = np.log(d[name])
    d['weight_sepsis'] = w*sep
    d['albumin_sepsis'] = d['albumin_g_l']*sep
    d['Watson_sepsis'] = d['Watson_TBW']*sep
    d['fat_mass'] = w - d['lean_bw_kg']
    d['lean_fraction'] = d['lean_bw_kg']/w
    d['crp_albumin_ratio'] = np.log1p(d['crp_mg_l'])/d['albumin_g_l']
    d['lactate_albumin_ratio'] = d['lactate_mmol_l']/d['albumin_g_l']
    return d

tr = add_bio_features(train)
te = add_bio_features(test)

for label, df in [('Train',tr),('Test',te)]:
    matches = (np.round(df['Janma_LBW'], 1) == df['lean_bw_kg']).sum()
    print(f'{label}: Janmahasatian, совпадений после округления {matches}/{len(df)}')


### Важное открытие

`lean_bw_kg` на всех 2500 строках (Train + Test) **точно** совпадает с формулой Janmahasatian при округлении до одной десятой. Это сильное свидетельство, что признак сгенерирован по формуле и не содержит независимого измерения. Но готовое значение иногда ускоряет обучение дерева.

In [ ]:
cols = ['CG_actual', 'CG_lean', 'CG_adjusted', 'CKDEPI_indexed',
        'CKDEPI_absolute', 'BMI', 'BSA', 'Watson_TBW', 'Janma_LBW',
        'lean_fraction', 'crp_albumin_ratio', 'lactate_albumin_ratio']
display(tr[cols + ['CL_L_h','Vd_L']].corr(method='spearman').loc[cols,['CL_L_h','Vd_L']].round(3))

## 2. Выводим эмпирическую формулу для CL

Для пациентов с известным креатинином оценим `log(CL) ~ log(CG_actual) + OAT`. Коэффициенты — вывод из конкурсного датасета, **не клинический стандарт**. Не используем пациенты из Test, там нет таргета.

In [ ]:
obs = tr.dropna(subset=['log_CG_actual'])
X = sm.add_constant(obs[['log_CG_actual','oat_rapid','oat_slow']])
fit = sm.OLS(np.log(obs['CL_L_h']), X).fit(cov_type='HC3')
display(pd.DataFrame({'coef':fit.params, 'lower95':fit.conf_int()[0], 'upper95':fit.conf_int()[1]}).round(4))
print('R² для log(CL), обучающие наблюдения:', round(fit.rsquared, 4))
print('Генетические множители: Rapid',round(np.exp(fit.params['oat_rapid']),3),
      'Slow',round(np.exp(fit.params['oat_slow']),3))
print('Мультипликативная модель: CL = A * CrCl^b * множитель OAT')

## 3. Валидация формул: CL

Честное сравнение: одни и те же 5 фолдов, обучение imputer и scaler только на train-фолде. Функция `log(Ridge)` предсказывает CL. Для чистоты сравнения сначала выводим **MAPE без посткалибровки** — поэтому числа могут отличаться от финального ансамбля, у которого есть калибровка и вспомогательная модель креатинина.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_simple(features, target, family='log_ridge'):
    y = tr[target].to_numpy()
    p = np.zeros(len(tr))
    for ti, vi in kf.split(tr):
        med = SimpleImputer(strategy='median')
        a = med.fit_transform(tr.iloc[ti][features])
        b = med.transform(tr.iloc[vi][features])
        scaler = StandardScaler()
        a = scaler.fit_transform(a)
        b = scaler.transform(b)
        if family == 'log_ridge':
            model = Ridge(alpha=2).fit(a, np.log(y[ti]))
            p[vi] = np.exp(model.predict(b))
        elif family == 'mape_quantile':
            # weighted LAD (вес 1/y) минимизирует среднюю процентную ошибку
            model = QuantileRegressor(quantile=.5, alpha=0, solver='highs')
            model.fit(a, y[ti], sample_weight=1/y[ti])
            p[vi] = np.maximum(.01, model.predict(b))
    ape = 100 * np.abs(y-p)/y
    return round(float(ape.mean()),4), p

cl_variants = {
    'CG, фактический вес + OAT':['log_CG_actual','oat_rapid','oat_slow'],
    'CG, скорректированный вес + OAT':['log_CG_adjusted','oat_rapid','oat_slow'],
    'CG, безжировая масса + OAT':['log_CG_lean','oat_rapid','oat_slow'],
    'CKD-EPI, абсолютная + OAT':['log_CKDEPI_absolute','oat_rapid','oat_slow'],
    'CKD-EPI, индексированная + OAT':['log_CKDEPI_indexed','oat_rapid','oat_slow'],
    'CG + CKD-EPI + OAT':['log_CG_actual','log_CKDEPI_absolute','oat_rapid','oat_slow'],
}
cl_results = [(name, cv_simple(cols, 'CL_L_h')[0]) for name,cols in cl_variants.items()]
display(pd.DataFrame(cl_results,columns=['CL formula','MAPE %']).sort_values('MAPE %'))

## 4. Валидация формул: Vd

Исследуем простые клинически интерпретируемые регрессии. Одинаковая 5-fold CV. Для MAPE используем медианную регрессию, взвешенную на `1/Vd_true` только в обучающем фолде. Проверим формулу Watson, взаимодействия с сепсисом и альбумином.

In [ ]:
vd_variants = {
    'Вес':['weight_kg'],
    'Вес + сепсис + взаимодействие':['weight_kg','sepsis_status','weight_sepsis'],
    'То же + альбумин':['weight_kg','sepsis_status','weight_sepsis','albumin_g_l'],
    'То же + альбумин×сепсис':['weight_kg','sepsis_status','weight_sepsis','albumin_g_l','albumin_sepsis'],
    'Watson TBW + сепсис + взаимодействие + альбумин':['Watson_TBW','sepsis_status','Watson_sepsis','albumin_g_l'],
    'Вес + Watson + сепсис + альбумин':['weight_kg','Watson_TBW','sepsis_status','weight_sepsis','Watson_sepsis','albumin_g_l'],
    'Вес + воспалительные индексы':['weight_kg','sepsis_status','weight_sepsis','albumin_g_l','crp_albumin_ratio','lactate_albumin_ratio'],
}
vd_results = [(name,cv_simple(cols,'Vd_L','mape_quantile')[0]) for name,cols in vd_variants.items()]
display(pd.DataFrame(vd_results,columns=['Vd formula','MAPE %']).sort_values('MAPE %'))

## 5. Проверяем, не случайно ли улучшение 0.02 процентного пункта

Сравниваем взвешенную медианную регрессию с альбумином и с дополнительным взаимодействием `albumin × sepsis`. Bootstrap по пациентам позволяет оценить разброс *наблюдаемой разности CV-ошибок*. Интервал включает ноль — уверенного улучшения нет.

In [ ]:
c1 = vd_variants['То же + альбумин']
c2 = vd_variants['То же + альбумин×сепсис']
_, p1 = cv_simple(c1, 'Vd_L', 'mape_quantile')
_, p2 = cv_simple(c2, 'Vd_L', 'mape_quantile')
y = tr.Vd_L.to_numpy()
delta = 100*(abs(y-p1)-abs(y-p2))/y
rng = np.random.default_rng(117)
bs = [delta[rng.integers(0,len(delta),len(delta))].mean() for _ in range(2000)]
print('Улучшение MAPE, процентных пунктов:',round(delta.mean(),4))
print('Приблизительный 95% bootstrap interval:',np.quantile(bs,[.025,.975]).round(4))

## 6. Выводы и границы применимости

1. **Janmahasatian LBW**: 2000/2000 совпадений в Train и 500/500 в Test. Это вычисляемый, а не независимый признак.
2. **CL**: среди проверенных формул для этого датасета лучше всего `Cockcroft–Gault` с фактической массой, хотя в клинике выбор веса зависит от обстоятельств. CKD-EPI — не «хуже вообще», только в данном соревновании при заданной модели.
3. **Vd**: вес, сепсис и их взаимодействие объясняют большую часть закономерности; альбумин добавляет небольшую полезную информацию.
4. **Watson, BMI, Janma**: биологически осмысленны, но улучшения MAPE по этому эксперименту не дали.
5. **Генетика OAT**: сильная наблюдаемая связь с CL именно в конкурсных данных; нельзя делать вывод, что такой же эффект существует для реального аминогликозида.
6. **Никакой утечки:** `patient_id`, `CL_L_h` и `Vd_L` не входят в созданные входные признаки, валидация выполняется по фолдам.
7. **Не используйте в медицинской практике:** CKD-EPI и Cockcroft–Gault особенно ограничены при нестабильной функции почек, как бывает в реанимации. Нужно клиническое подтверждение и мониторинг концентраций реальных препаратов.

Это ноутбук для **поиска и проверки гипотез**, а не для замены итогового ансамбля; на уже проверенном итоговом ансамбле добавление группы биопризнаков не улучшило качество.
